In [ ]:
import matplotlib.pylab as plt
import xarray as xr
import pint_xarray
import numpy as np
import cftime

ds_hy = xr.open_dataset("/Users/andy/base/pism-terra/2026_07_ismip7_dev_hy/output/scalar/scalar_g1200m_id_CESM2-WACCM_historical_1985-01-01_2015-01-01.nc").expand_dims({"uq": ["HYBRID"]})
ds_ho = xr.open_dataset("/Users/andy/base/pism-terra/2026_07_ismip7_dev_ho/output/scalar/scalar_g1200m_id_CESM2-WACCM_historical_1985-01-01_2015-01-01.nc").expand_dims({"uq": ["HO"]})
ds = xr.concat([ds_hy, ds_ho], dim="uq", join="outer").pint.quantify().convert_calendar("standard", use_cftime=False)

grace = xr.open_dataset("/Users/andy/base/pism-ragis/data/grace/greenland_mass_balance.nc").squeeze().pint.quantify()
mankoff = xr.open_dataset("/Users/andy/base/pism-ragis/data/mass_balance/mankoff_greenland_mass_balance_clean.nc").pint.quantify()
mankoff = mankoff.sum(dim="region").resample(time='YE').mean('time').pint.to("Gt/yr")

sigma = 2
mankoff_mb = mankoff.MB
mankoff_mb_err = mankoff.MB_err
mankoff_smb = mankoff.SMB
mankoff_glf = -mankoff.D


In [ ]:
mass = ds.ice_mass_glacierized
mass = mass - mass.sel(time="2002", method="nearest")
mass = mass.pint.to("Gt")

mass = (ds.tendency_of_ice_mass.pint.to("Gt/yr") - xr.DataArray(400).pint.quantify("Gt/yr")).cumsum(dim="time") 
mass = mass - mass.sel(time="2002", method="nearest")

fig, ax = plt.subplots(1, 1)
mass.plot(hue="uq", ax=ax)
grace.cumulative_mass_balance.plot(ax=ax)
ax.set_xlim(np.datetime64("1985"), np.datetime64("2015"))

mb = ds.tendency_of_ice_mass_glacierized.pint.to("Gt yr^-1")
glf = ds.grounding_line_flux.pint.to("Gt yr^-1")
smb = ds.tendency_of_ice_mass_due_to_surface_mass_flux.pint.to("Gt yr^-1")

uq_vals = mb["uq"].values
palette = dict(zip(uq_vals, plt.cm.Paired(np.linspace(0, 1, len(uq_vals)))))

fig, axs = plt.subplots(3, 1, figsize=(6.4, 10.4))
mankoff_mb.plot(ax=axs[0], color="r", lw=2)
mankoff_smb.plot(ax=axs[1], color="r", lw=2)
mankoff_glf.plot(ax=axs[2], color="r", lw=2)
for k, (da, ls) in enumerate([(mb, "solid"), (smb, "dotted"), (glf, "dashed")]):
    ax = axs[k]
    for u in da["uq"].values:
        da.sel(uq=u).plot(ax=ax, color=palette[u], ls=ls)
    ax.set_title(None)
    ax.axhline(0, color="k", lw=0.5, ls="dotted")
    ax.set_xlim(np.datetime64("1985"), np.datetime64("2015"))

#axs[0].fill_between(mankoff_mb - sigma * mankoff_mb_err, mankoff_mb + sigma * mankoff_mb_err, alpha=0.5, color="r")


In [ ]:
mankoff.sum(dim="region").SMB.resample(time='YE').mean('time').pint.to("Gt/yr").plot()

In [ ]:
mankoff_mb - sigma * mankoff_mb_err

In [ ]:
ax.fill_between(mankoff_mb - sigma * mankoff_mb_err, mankoff_mb + sigma * mankoff_mb_err, alpha=0.5, color="r")

In [ ]:
plt.show()

In [ ]:
mankoff_mb_err